Search for matching candidates given a job profile.

Requires both Elasticsearch indices:
- `job_profiles` (job embeddings)
- `candidate_profiles` (candidate embeddings)

In [1]:
%load_ext autoreload
%autoreload 2

import importlib
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import core.elasticsearch
import core.search
importlib.reload(core.elasticsearch)
importlib.reload(core.search)

from core.config import EmbeddingConfig, ElasticsearchConfig
from core.elasticsearch import get_es_client
from core.indices import CANDIDATE_INDEX, JOB_INDEX
from core.search import (
    search_candidates_by_job_id,
    search_candidates_by_profile_text,
)

K = 10
NUM_CANDIDATES = 100
BATCH_JOBS = 10  # for optional batch export
OUTPUT_PATH = Path('../data/job_candidate_matches.jsonl')

embed_config = EmbeddingConfig.from_env()
es = get_es_client(ElasticsearchConfig.from_env())

/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/elasticsearch/_sync/client/__init__.py:328: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [2]:
for index in (JOB_INDEX, CANDIDATE_INDEX):
    count = es.count(index=index)['count']
    print(f'{index}: {count:,} documents')

job_profiles: 141,897 documents
candidate_profiles: 210,250 documents


/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [8]:
# Pick a sample job id from the index
sample = es.search(index=JOB_INDEX, size=1, _source=['id', 'position', 'primary_keyword'])
job_hit = sample['hits']['hits'][0]
job_id = job_hit['_id']
job_source = job_hit['_source']

print(f"Sample job id: {job_id}")
print(f"Position: {job_source.get('position', '')!r}")
print(f"Keyword: {job_source.get('primary_keyword', '')!r}")

Sample job id: c0ca96e7-85df-50df-a64e-d934cd02a170
Position: '10 + Blockchain Nodes / Masternodes to set up'
Keyword: 'Sysadmin'


/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
# Search by stored job embedding (no API call)
# job_id = '246364be-ac7b-59b3-bdf2-fa7e6c10a9f0' # to search by specific job id

hits = search_candidates_by_job_id(
    es,
    job_id,
    k=K,
    num_candidates=NUM_CANDIDATES,
)

print(f'Top {K} candidates for job {job_id}:')
for hit in hits:
    print(
        f"  score={hit['score']:.4f}  "
        f"id={hit['id']}  "
        f"position={hit.get('position', '')!r}  "
        f"keyword={hit.get('primary_keyword', '')!r}"
    )

Top 10 candidates for job 246364be-ac7b-59b3-bdf2-fa7e6c10a9f0:
  score=0.9025  id=4a3d7797-1989-5866-b3c9-841a092e709f  position='Таргетолог'  keyword='Marketing'
  score=0.8973  id=79877117-5b94-5bde-91a4-1252ebca25f8  position='Таргетолог | Targetologist'  keyword='Marketing'
  score=0.8921  id=fa042dc9-a68c-5851-82e0-58495cc1244e  position='Таргетолог'  keyword='Marketing'
  score=0.8890  id=dfc60686-8aa1-5baa-a606-14310cc510ff  position='Таргетолог, смм'  keyword='Marketing'
  score=0.8856  id=50d098a3-5ea4-5934-ad62-05a4bab74823  position='Таргетолог- маркетолог'  keyword='Marketing'
  score=0.8853  id=534e64b9-fe89-5166-b402-fffcb0550ae1  position='Контентмейкер, Таргетолог, Сторісмейкер, Ведення соц. мереж'  keyword='Marketing'
  score=0.8783  id=284137c1-8cbd-5af7-a395-187d0658f1e4  position='Таргетолог'  keyword='Other'
  score=0.8773  id=f47a4547-a6e5-5b64-b55d-fc8b2f8ad33b  position='Targetolog "FB+INST (ppc, f'  keyword='Lead Generation'
  score=0.8754  id=93634498-70a0-55

/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
# Search by job profile_text (embed at query time)
job_doc = es.get(index=JOB_INDEX, id=job_id)
profile_text = job_doc['_source']['profile_text']

hits_live = search_candidates_by_profile_text(
    profile_text,
    es,
    k=K,
    num_candidates=NUM_CANDIDATES,
    config=embed_config,
)

print(f'Top {K} candidates via live embed:')
for hit in hits_live:
    print(
        f"  score={hit['score']:.4f}  "
        f"id={hit['id']}  "
        f"position={hit.get('position', '')!r}"
    )

In [ ]:
# Optional: batch top-k matches for first N jobs
job_ids = [
    hit['_id']
    for hit in es.search(index=JOB_INDEX, size=BATCH_JOBS, _source=False)['hits']['hits']
]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w') as f:
    for jid in job_ids:
        matches = search_candidates_by_job_id(
            es,
            jid,
            k=K,
            num_candidates=NUM_CANDIDATES,
        )
        record = {
            'job_id': jid,
            'candidates': [
                {'candidate_id': m['id'], 'score': m['score']} for m in matches
            ],
        }
        f.write(json.dumps(record) + '\n')

print(f'Wrote {len(job_ids)} job match records to {OUTPUT_PATH}')